In [1]:
import pandas as pd
import os
import sys
sys.path.append("../")

from data.dataset import SokobanDataset, act, play, parse_sokoban_level, compile_sokoban_state, symbolic_state_to_tensor

config_dataset = {
    "source_levels": {
        "github": "google-deepmind/boxoban-levels",
        "cache_dir": "~/scratch/curry/",
    },
    "source_solutions": {
        "huggingface": "AlignmentResearch/boxoban-astar-solutions",
        "cache_dir": "~/scratch/curry/",
    },

    "difficulty": "unfiltered",
    "split": "train",
    "grid_shape": {
        "x": 10,
        "y": 10
    },
    "max_num_levels": 8,
    "solution_length":{
        "min": -1,
        "max": 1000,
    },
    "order_by": "shuffle",
    "seed": 42,
}
dataset = SokobanDataset(config_dataset)
next(iter(dataset))

       boxes    sol_len      walls
count    8.0   8.000000   8.000000
mean     4.0  29.000000  67.125000
std      0.0  13.255727   4.764077
min      4.0  10.000000  62.000000
25%      4.0  21.500000  63.750000
50%      4.0  27.000000  65.000000
75%      4.0  37.000000  71.500000
max      4.0  50.000000  74.000000
Not solved stats: {}


{'level_str': '##########\n##########\n##########\n#####.  ##\n# $..    #\n#@   . $ #\n# $ ###$ #\n#    ##  #\n#     ## #\n##########\n',
 'actions_str': '01113323212100110111231223003110333',
 'folder_name': 524,
 'level_name': 602,
 'states_tensor': tensor([[[[0., 0., 0., 1.],
           [0., 0., 0., 1.],
           [0., 0., 0., 1.],
           ...,
           [0., 0., 0., 1.],
           [0., 0., 0., 1.],
           [0., 0., 0., 1.]],
 
          [[0., 0., 0., 1.],
           [0., 0., 0., 1.],
           [0., 0., 0., 1.],
           ...,
           [0., 0., 0., 1.],
           [0., 0., 0., 1.],
           [0., 0., 0., 1.]],
 
          [[0., 0., 0., 1.],
           [0., 0., 0., 1.],
           [0., 0., 0., 1.],
           ...,
           [0., 0., 0., 1.],
           [0., 0., 0., 1.],
           [0., 0., 0., 1.]],
 
          ...,
 
          [[0., 0., 0., 1.],
           [0., 0., 0., 0.],
           [0., 0., 0., 0.],
           ...,
           [0., 0., 0., 0.],
           [0., 0., 0

In [2]:
from torch.utils.data import DataLoader
from data.dataset import collate_fn

dataloader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

In [3]:
from models.thinker import Thinker
from models.consolidator import Consolidator
import torch
from torch import nn

args = {
    "num_think_steps": 1,
    "num_supervision_steps": 2,
}
model_config = {
    "thinker_config": {
        "config_visual_encoder": {
            "in_channels": 4,
            "latent_dim": 64,
            "grid_shape_x": 10,
            "grid_shape_y": 10,
            "channels": [16, 32, 64],   # out channels for each conv layer
            "kernel_size": 3,
            "padding": 1,
            "stride": 2,                # applied only to last conv
        },
        "config_action_decoder": {
            "model_name": "qwen2",
            "args": {
                "num_layers": 2,
                "hidden_size": 64,
                "num_attention_heads": 4,
            },
            "hidden_size": 64,
            "vocab_size": 4,
            "num_think_steps": args["num_think_steps"],
        },
    },
    "consolidator_config": {
        "model_name": "t5",
        "args": {
            "num_encoder_layers": 1,
            "num_decoder_layers": 1,
            "hidden_size": 64,
            "num_attention_heads": 2,
        }
    },
    "memory_config": {
        "hidden_size": 64,
        "memory_size": 2,
    },
    "num_supervision_steps": args["num_supervision_steps"],
}

class MAPLE(nn.Module):
    def __init__(self, config):
        super().__init__()

        thinker_config = config["thinker_config"]
        self.thinker = Thinker(thinker_config)
        consolidator_config = config["consolidator_config"]
        self.consolidator = Consolidator(consolidator_config)

        # Memory as nn.Parameter
        memory_config = config["memory_config"]
        self.memory = nn.Parameter(torch.randn(memory_config["memory_size"], memory_config["hidden_size"]))

        # Supervision
        self.num_supervision_steps = config["num_supervision_steps"] 
        
    def forward(self, batch):

        B = batch["states_tensors"].size(0)
        
        memory_states = self.memory.unsqueeze(0).expand(B, -1, -1)  # shape [B, memory_size, hidden_size]
        thinker_outputs = []
        for _ in range(self.num_supervision_steps):
            thinker_output = self.thinker(batch, memory_states)
            memory_states = self.consolidator({
                "memory_states": memory_states,
                "thinking_stream": thinker_output["decoder_output"]["last_hidden_state"],
            })
            thinker_outputs.append(thinker_output)
        return thinker_outputs

model = MAPLE(model_config)
model.eval()

MAPLE(
  (thinker): Thinker(
    (visual_encoder): VisualEncoder(
      (cnn): Sequential(
        (0): Conv2d(4, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU()
        (2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU()
        (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (5): ReLU()
      )
      (fc): Linear(in_features=1600, out_features=64, bias=True)
    )
    (action_decoder): ActionDecoder(
      (backbone): Qwen2Model(
        (embed_tokens): Embedding(151936, 64)
        (layers): ModuleList(
          (0-1): 2 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): Linear(in_features=64, out_features=64, bias=True)
              (k_proj): Linear(in_features=64, out_features=64, bias=True)
              (v_proj): Linear(in_features=64, out_features=64, bias=True)
              (o_proj): Linear(in_features=64, out_features=64, bias=False)
         

In [ ]:
def autoregressive_thinker(model_thinker, dynamic_batch, states, memory_states, max_solution_length=100):
    B = len(states)
    last_status = ["in progress"] * B
    with torch.no_grad():
        for _ in range(max_solution_length):
        
            thinker_output = model_thinker(dynamic_batch, memory_states)

            logits = thinker_output["decoder_output"]["logits"]
            preds = torch.argmax(logits, dim=-1)
            
            new_states = []
            new_states_tensor = []
            new_attention_mask = []
            for b, a in zip(range(B), preds[:, -1]):
                state = states[b]
                action_str = str(a.item())
                if dynamic_batch["attention_mask"][b, -1] == 0:
                    new_states.append(state)
                    new_states_tensor.append(torch.zeros_like(dynamic_batch["states_tensors"][b, :1]))
                    new_attention_mask.append([0])
                else:
                    new_state, status = act(state, action_str)
                    last_status[b] = status
                    if status == "in progress":
                        new_state_tensor = symbolic_state_to_tensor(new_state, grid_shape_x, grid_shape_y, channels).unsqueeze(0)
                    else:
                        new_state_tensor = torch.zeros_like(dynamic_batch["states_tensors"][b, :1]) 
                    new_states.append(new_state)
                    new_states_tensor.append(new_state_tensor)
                    new_attention_mask.append([1] if status == "in progress" else [0])

            states = new_states
            new_attention_mask = torch.tensor(new_attention_mask)
            if new_attention_mask.max() == 0: break

            new_states_tensor = torch.stack(new_states_tensor, dim=0)
            
            dynamic_batch["states_tensors"] = torch.cat([dynamic_batch["states_tensors"], new_states_tensor], dim=1)
            dynamic_batch["attention_mask"] = torch.cat([dynamic_batch["attention_mask"], new_attention_mask], dim=1)

    return thinker_output

def get_dynamic_batch(level_strs):
    dynamic_batch = {}
    states = [parse_sokoban_level(level_str) for level_str in level_strs]
    states_tensor = torch.stack([
        symbolic_state_to_tensor(state, grid_shape_x, grid_shape_y, channels).unsqueeze(0)
        for state in states
    ])
    batch_attention_mask = torch.tensor([[1]*1 for _ in range(len(level_strs))])  # Dummy attention mask for 1 think step
    dynamic_batch["states_tensors"] = states_tensor
    dynamic_batch["attention_mask"] = batch_attention_mask
    return dynamic_batch, states

channels = ['boxes', 'goals', 'player', 'walls']
grid_shape_x = config_dataset["grid_shape"]["x"]
grid_shape_y = config_dataset["grid_shape"]["y"]

batch = next(iter(dataloader))

dynamic_batch, states_0 = get_dynamic_batch(batch["level_strs"])

memory_states = model.memory.unsqueeze(0).expand(len(states_0), -1, -1)  # shape [B, memory_size, hidden_size]
thinker_outputs = []
with torch.no_grad():
    for _ in range(model.num_supervision_steps):
        thinker_output = autoregressive_thinker(model.thinker, dynamic_batch, states_0, memory_states, max_solution_length=20)
        memory_states = model.consolidator({
            "memory_states": memory_states,
            "thinking_stream": thinker_output["decoder_output"]["last_hidden_state"],
            "thinking_stream_attention_mask": thinker_output["attention_mask"],
        })
        thinker_outputs.append(thinker_output)